# 04 · LangChain implementation

The notebook 03 pipeline (same index, models, HyDE, hybrid search, re-ranking and abstain gate)
implemented with LangChain, extended with a streaming chat interface with conversation memory. Reads
the index built in notebook 02 without modifying it.

| Component | Notebook 03 | Notebook 04 |
|---|---|---|
| Raw papers | dicts from `papers.jsonl` | `Document(page_content, metadata)` |
| Mixed chunking | `mixed_chunk()` | `RecursiveCharacterTextSplitter` |
| Bi-encoder | `SentenceTransformer` | `HuggingFaceEmbeddings` |
| Vector DB + hybrid search | `collection.query.hybrid(...)` | `WeaviateVectorStore` (`langchain-weaviate`) |
| LLM | `transformers` + `llm_chat()` | `HuggingFacePipeline` + `ChatHuggingFace`; 9B for answers, 2B for internal steps |
| Prompts | f-strings | `ChatPromptTemplate` |
| HyDE query parsing | `parse_query()` + `hyde_vector()` | `hyde` chain (`hyde_chain` → `hyde_vector`) |
| Re-ranker | `CrossEncoder` | `BaseCrossEncoder` (`langchain-core`) over `sentence-transformers` |
| Chat with memory | none | query rewriting + `SummarizationMiddleware` + streaming chat UI (section 10) |
| Orchestration | `ask()` | LCEL (`\|`, `RunnablePassthrough`, `RunnableBranch`) |

**Requirements.** LangChain is not part of `requirements.txt`:
```bash
pip install langchain==1.4.7 langchain-weaviate==0.0.8
pip install --no-deps langchain-huggingface==1.2.3
```
`langchain-huggingface` pins `huggingface-hub<2.0`; `--no-deps` keeps the hub 2.x required by
`transformers`. The local-pipeline classes used here work with hub 2.x (`pip check` still reports the
pin). `langchain-community` is not used.

**GPU memory.** The two LLMs (9B + 2B), embedder and re-ranker need ~25 GB. Run without other ArxivSage
kernels; kernels also share the embedded Weaviate instance.

In [ ]:
import functools
import html
import json
import time
from operator import itemgetter

import ipywidgets as widgets
import mistune
import numpy as np
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from weaviate.classes.query import Filter, HybridFusion

from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.cross_encoders import BaseCrossEncoder
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage, get_buffer_string
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_weaviate import WeaviateVectorStore

from common import *

## 1. Documents and chunking
Papers are represented as LangChain `Document`s (`page_content` + `metadata`).
`RecursiveCharacterTextSplitter` splits on the coarsest separator first (paragraph, line, sentence),
falls back to finer separators for oversized pieces, and merges pieces up to `chunk_size` with
`chunk_overlap`.

Configured to match notebook 02: a word-count `length_function` (`chunk_size=120`, `chunk_overlap=12`)
and `keep_separator="end"`, so punctuation stays with its sentence. Applied to three papers for
illustration; the full index already exists.

In [ ]:
papers = [json.loads(line) for _, line in zip(range(3), open(PAPERS_PATH))]
docs = [Document(page_content=p["abstract"],
                 metadata={"arxiv_id": p["arxiv_id"], "title": p["title"]}) for p in papers]

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=CHUNK_WORDS,
    chunk_overlap=OVERLAP_WORDS,
    length_function=lambda text: len(text.split()),
    keep_separator="end",
)
chunks = splitter.split_documents(docs)
for c in chunks:
    print(f"[{c.metadata['arxiv_id']}] {len(c.page_content.split())} words | {c.page_content[:90]}...")

**Indexing with LangChain** (not executed):
```python
vectorstore = WeaviateVectorStore.from_documents(
    chunks, embeddings, client=client, index_name="ArxivChunk", text_key="text")
```
`from_documents` creates the collection schema automatically, without control over HNSW parameters,
BM25 `k1`/`b`, property data types or batching. Notebook 02 creates the collection explicitly with the
Weaviate client.

## 2. Embeddings
`HuggingFaceEmbeddings` (sentence-transformers) with separate encode settings for documents and queries;
the BGE instruction prefix is applied to queries only.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda", "model_kwargs": {"dtype": torch.float16}},
    encode_kwargs={"normalize_embeddings": True},                                # documents
    query_encode_kwargs={"prompt": QUERY_INSTRUCTION, "normalize_embeddings": True},  # queries
)
print(len(embeddings.embed_query("test")), "dimensions")

## 3. Vector store
`WeaviateVectorStore` wraps the existing `ArxivChunk` collection (`text_key="text"`; all other
properties become `Document.metadata`). Its search methods call Weaviate's `hybrid()` and forward extra
keyword arguments, which carries over `alpha`, the fusion type, `title^2` boosting, filters and a
precomputed query vector.

The plain retriever below can return several chunks of the same paper; the full pipeline keeps one
chunk per paper after re-ranking.

In [ ]:
client = get_weaviate_client()
vectorstore = WeaviateVectorStore(client=client, index_name=COLLECTION, text_key="text", embedding=embeddings)

ALPHA = 0.6
N_CANDIDATES = 50
HYBRID_KWARGS = dict(alpha=ALPHA, fusion_type=HybridFusion.RELATIVE_SCORE, query_properties=["title^2", "text"])

# Standard LangChain retriever: plain hybrid search, no HyDE, no re-ranking.
retriever = vectorstore.as_retriever(search_kwargs={"k": 5, **HYBRID_KWARGS})
for d in retriever.invoke("speculative decoding for faster LLM inference"):
    print(f"  [{d.metadata['arxiv_id']}] {d.metadata['title'][:90]}")
# Several chunks of one paper can be returned; the full pipeline de-duplicates after re-ranking.

## 4. LLMs
`HuggingFacePipeline` wraps a local `transformers` text-generation pipeline; `ChatHuggingFace` adds chat
messages through the model's chat template. `ChatHuggingFace` does not forward chat-template options,
so `enable_thinking=False` is bound to the tokenizer with `functools.partial`. All calls decode greedily.

### Model split
Generation dominates latency, and HyDE generates ~200–250 tokens per question that are never shown.
HyDE, query rewriting and summaries therefore run on `FAST_LLM_MODEL` (Qwen3.5-2B); the 9B generates
answers only.

Evaluation on 8 questions, each model's HyDE output passed through the same hybrid search and re-ranker:

| HyDE model | HyDE time | Mean rerank score (top 6) | Top-6 overlap with 9B |
|---|---|---|---|
| 9B | 9.1 s | 0.949 | n/a |
| 2B, all filters | 3.8 s | 0.933 | 2.8 / 6 |
| **2B, year filters only** | **3.8 s** | **0.946** | **4.4 / 6** |

The 2B abstracts retrieve as well as the 9B ones, but its category predictions were unreliable (LLM, RAG
and LoRA questions tagged `cs.CV`, excluding the relevant NLP papers). With category filtering disabled,
the 2B saves ~5 s per question at equal retrieval quality.

In [ ]:
def make_chat(model_id, max_new_tokens):
    """Load a local Hugging Face model as a LangChain chat model (greedy decoding, thinking mode off)."""
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.apply_chat_template = functools.partial(tok.apply_chat_template, enable_thinking=False)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="cuda:0")
    generator = pipeline("text-generation", model=model, tokenizer=tok,
                         max_new_tokens=max_new_tokens, do_sample=False, return_full_text=False)
    return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator, model_id=model_id), tokenizer=tok)

chat = make_chat(LLM_MODEL, max_new_tokens=800)            # 9B: the answers users read
fast_chat = make_chat(FAST_LLM_MODEL, max_new_tokens=500)  # 2B: HyDE, query rewriting, summaries

print(chat.invoke("In one sentence, what is retrieval-augmented generation?").content)

## 5. HyDE
LangChain's `HypotheticalDocumentEmbedder` (`langchain_classic`) embeds only the hypothetical document
and cannot extract filters. HyDE is implemented as three LCEL chains instead:

| Chain | Function | Output |
|---|---|---|
| `hyde_chain` | prompt → 2B → `JsonOutputParser` | `plan`: hypothetical abstract + year range |
| `hyde_vector` | `RunnableParallel` embeds the question (BGE query prefix) and the abstract concurrently, then averages | `vector`: grounded HyDE vector |
| `hyde` | `hyde_chain` followed by `hyde_vector`, each adding its key to the input dict | `{question, plan, vector}` |

`with_fallbacks` returns `{}` on invalid JSON; `hyde_vector` then embeds the question in place of the abstract.

In [ ]:
HYDE_PARSE_SYSTEM = """You are the query parser of a search engine over arXiv papers in computer vision (cs.CV) and NLP (cs.CL), submitted between {start} and {end}.
Given a user question, return a JSON object with exactly these keys:
- "hypothetical_abstract": a plausible 120-180 word arXiv abstract of a paper that would answer the question, written in the typical style of the field (problem, method, results). Use the technical terminology a real paper would use.
- "year_from": integer or null. Only set if the user explicitly restricts the time range (e.g. "since 2025", "recent work in 2026").
- "year_to": integer or null. Same rule.
Return only the JSON object."""

hyde_parse_prompt = ChatPromptTemplate.from_messages([
    ("system", HYDE_PARSE_SYSTEM),
    ("human", "{question}"),
]).partial(start=str(START_DATE.year), end=str(END_DATE.year))

hyde_chain = (hyde_parse_prompt | fast_chat | JsonOutputParser()).with_fallbacks([RunnableLambda(lambda _: {})])

def normalized_mean(vecs):
    v = np.add(vecs["question"], vecs["abstract"])
    return (v / np.linalg.norm(v)).tolist()

hyde_vector = RunnableParallel(
    question=itemgetter("question") | RunnableLambda(embeddings.embed_query),             # query: BGE prefix
    abstract=RunnableLambda(lambda x: [x["plan"].get("hypothetical_abstract") or x["question"]])
             | embeddings.embed_documents | itemgetter(0),                                # document: no prefix
) | normalized_mean

hyde = RunnablePassthrough.assign(plan=hyde_chain) | RunnablePassthrough.assign(vector=hyde_vector)

out = hyde.invoke({"question": "How do recent papers reduce hallucination in vision-language models?"})
print(json.dumps(out["plan"], indent=2))
print(f"\nvector: {len(out['vector'])} dims, norm {np.linalg.norm(out['vector']):.3f}")

## 6. Retrieval
Converts the year range into a Weaviate `Filter` and runs hybrid search: the HyDE vector drives the
vector side, the original question drives BM25. No category filter (see section 4).

In [ ]:
def build_filters(year_from=None, year_to=None):
    parts = []
    if year_from:
        parts.append(Filter.by_property("year").greater_or_equal(int(year_from)))
    if year_to:
        parts.append(Filter.by_property("year").less_or_equal(int(year_to)))
    return Filter.all_of(parts) if parts else None

def retrieve(inputs):
    """{question, plan, vector} -> 50 candidate Documents, each with metadata['hybrid_score']."""
    plan = inputs["plan"]
    hits = vectorstore.similarity_search_with_score(
        inputs["question"], k=N_CANDIDATES, vector=inputs["vector"],
        filters=build_filters(plan.get("year_from"), plan.get("year_to")),
        **HYBRID_KWARGS,
    )
    for doc, score in hits:
        doc.metadata["hybrid_score"] = score
        doc.metadata["published"] = doc.metadata["published"].strftime("%Y-%m")
    return [doc for doc, _ in hits]

## 7. Re-ranking
LangChain's standard re-ranking path is
```python
ContextualCompressionRetriever(base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=6),
                               base_retriever=retriever)
```
`CrossEncoderReranker` discards the scores (required by the abstain gate) and does not de-duplicate by
paper, so re-ranking is a custom step that stores each score in `Document.metadata` and keeps the best
chunk per paper.

The cross-encoder implements `langchain_core.cross_encoders.BaseCrossEncoder` over `sentence-transformers`
and remains compatible with `CrossEncoderReranker`.

In [ ]:
TOP_K_PAPERS = 6
MIN_RERANK_SCORE = 0.05

class STCrossEncoder(BaseCrossEncoder):
    """LangChain cross-encoder interface backed by a local sentence-transformers model."""
    def __init__(self, model_name, **kwargs):
        self.model = CrossEncoder(model_name, **kwargs)

    def score(self, text_pairs):
        return self.model.predict(text_pairs, batch_size=64, show_progress_bar=False).tolist()

cross_encoder = STCrossEncoder(RERANK_MODEL, device="cuda", max_length=384, model_kwargs={"dtype": torch.float16})

def rerank(inputs):
    """{question, candidates} -> top 6 Documents (best chunk per paper), with metadata['rerank_score']."""
    question, candidates = inputs["question"], inputs["candidates"]
    if not candidates:
        return []
    scores = cross_encoder.score([(question, f"{d.metadata['title']}\n{d.page_content}") for d in candidates])
    for d, s in zip(candidates, scores):
        d.metadata["rerank_score"] = float(s)
    best = {}
    for d in sorted(candidates, key=lambda d: -d.metadata["rerank_score"]):
        best.setdefault(d.metadata["arxiv_id"], d)
    return list(best.values())[:TOP_K_PAPERS]

## 8. Answer chain and pipeline
Each `RunnablePassthrough.assign` step adds its result to the dict flowing through the chain:

```
{question}
  retrieval_chain:
  → hyde                assign(plan): fake abstract + years (2B), then assign(vector): grounded HyDE vector
  → assign(candidates)  hybrid search + filters       (Weaviate)
  → assign(sources)     re-ranking                    (cross-encoder)
  then:
  → RunnableBranch:     abstain gate                  best rerank score < 0.05 → fixed reply, no LLM call
                        otherwise → assign(answer)    (9B)
```

`retrieval_chain` is a separate chain so the chat interface can run retrieval and stream the answer step alone.

In [ ]:
ANSWER_SYSTEM = """You are ArxivSage, a research assistant for computer vision and NLP.
Answer the user's question using ONLY the numbered arXiv sources provided.
Rules:
- Cite every factual claim with the source number in square brackets, e.g. [1] or [2][4].
- Synthesize: group papers by approach and compare them, rather than listing them one by one.
- Be concise: 1 short intro sentence, then 3-6 bullet points, then a one-line takeaway.
- If the sources do not contain enough information to answer, say exactly what is missing. Never use outside knowledge and never invent papers or numbers."""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", ANSWER_SYSTEM),
    ("human", "Sources:\n\n{context}\n\nQuestion: {question}"),
])

def format_sources(docs):
    return "\n\n".join(
        f"[{i}] {d.metadata['title']} (arXiv:{d.metadata['arxiv_id']}, {d.metadata['published']}, "
        f"{d.metadata['primary_category']})\n{d.page_content}"
        for i, d in enumerate(docs, 1))

answer_chain = (
    {"context": itemgetter("sources") | RunnableLambda(format_sources), "question": itemgetter("question")}
    | answer_prompt | chat | StrOutputParser()
)

ABSTAIN_MESSAGE = ("I couldn't find papers in the indexed CV/NLP arXiv corpus "
                   f"({START_DATE.year}–{END_DATE.year}) that address this question.")

def is_weak(x):
    return not x["sources"] or x["sources"][0].metadata["rerank_score"] < MIN_RERANK_SCORE

retrieval_chain = (
    hyde                                                               # HyDE: plan (fake abstract + years) + grounded vector
    | RunnablePassthrough.assign(candidates=RunnableLambda(retrieve))  # hybrid search with that vector + filters
    | RunnablePassthrough.assign(sources=RunnableLambda(rerank))
)

rag_chain = (
    retrieval_chain
    | RunnableBranch(
        (is_weak, RunnablePassthrough.assign(answer=lambda _: ABSTAIN_MESSAGE, sources=lambda _: [])),
        RunnablePassthrough.assign(answer=answer_chain),
    )
)

## 9. Examples

In [ ]:
def show(result):
    md = [f"### Q: {result['question']}", result["answer"]]
    if result["sources"]:
        md.append("\n**Sources**")
    for i, d in enumerate(result["sources"], 1):
        m = d.metadata
        md.append(f"{i}. [{m['title']}]({m['url']}) · {m['published']} · {m['primary_category']} "
                  f"· rerank={m['rerank_score']:.2f}")
    display(Markdown("\n\n".join(md)))

show(rag_chain.invoke({"question": "How do recent papers reduce hallucination in vision-language models?"}))

In [ ]:
show(rag_chain.invoke({"question": "What is the best recipe for sourdough bread?"}))   # should abstain

## 10. Chat with memory
Follow-up questions are handled in two parts:

1. **Query rewriting.** A follow-up such as "Which of those need no extra training?" is rewritten into a
   standalone question using the conversation; retrieval and answering run on the rewritten question.
2. **History summarization.** `SummarizationMiddleware` replaces older turns with a single summary
   message once the history reaches `trigger` messages, keeping the last `keep` messages verbatim.

```
 each turn (session_id = one conversation)
   history = sessions[session_id]                      (summary message + recent turns)
   1. rewrite:  history + follow-up → standalone question      (2B, skipped on turn 1)
   2. answer:   retrieval_chain → answer_chain.stream(...)     (HyDE on 2B → search → rerank → 9B answer)
   3. save:     history + [question, answer + source titles]
   4. compact:  summarizer: ≥ trigger messages? → [summary] + last `keep` messages   (2B, only when triggered)
```

**Design notes**
- The answer step does not receive the history. With history, the model reused claims from earlier
  answers instead of the new sources, and earlier citation numbers referred to a different source list.
- The rewriter receives the history as text (`get_buffer_string`) with two examples and explicit rules
  for references ("those", "ones", "it"); otherwise the small model tends to return follow-ups unchanged.
- Stored answers include their source titles, so references such as "the second paper" can be resolved.
- `SummarizationMiddleware` is designed as an agent hook; its `before_model` method operates on a plain
  `{"messages": [...]}` dict and is called directly. (`ConversationSummaryBufferMemory` is deprecated.)
- Sessions are held in memory and do not persist across kernel restarts.

In [ ]:
SUMMARY_PROMPT = """Summarize this conversation between a user and ArxivSage, a research assistant over arXiv computer vision and NLP papers.
Keep: the topics and questions the user asked, the methods and paper titles mentioned in the answers, and any constraints the user set (years, fields).
Write 3-6 short bullet points. Return only the summary.

<messages>
{messages}
</messages>"""

summarizer = SummarizationMiddleware(
    model=fast_chat,             # the 2B writes the summary
    trigger=("messages", 6),     # summarize once the history has 6 messages (3 turns)...
    keep=("messages", 2),        # ...keeping the latest question/answer pair word for word
    summary_prompt=SUMMARY_PROMPT,
)

CONDENSE_SYSTEM = """You rewrite follow-up questions for a search engine over arXiv papers.
The search engine cannot see the conversation, so the rewritten question must name its topic explicitly.
Replace every reference to the conversation ("those", "them", "ones", "it", "these methods", "the second paper") with what it refers to, and keep all technical terms and constraints (years, fields).
Always include the main research topic of the conversation (e.g. "hallucination in vision-language models") in the standalone question.
Only if the follow-up already names its full topic, return it unchanged. Return only the question.

Examples:
Conversation: Human: How do LLMs handle long contexts? AI: They use sparse attention [1] and memory tokens [2].
Follow-up: Which of these are cheapest at inference?
Standalone question: Which long-context methods for LLMs, such as sparse attention or memory tokens, are cheapest at inference?

Conversation: Human: What are recent methods for 3D reconstruction from a single image? AI: Diffusion priors [1] and Gaussian splatting [2].
Follow-up: Only ones from 2025.
Standalone question: What are methods from 2025 for 3D reconstruction from a single image?"""

condense_chain = ChatPromptTemplate.from_messages([
    ("system", CONDENSE_SYSTEM),
    ("human", "Conversation:\n{history}\n\nFollow-up question: {question}\n\nStandalone question:"),
]) | fast_chat | StrOutputParser()   # the 2B rewrites follow-ups

sessions = {}   # session_id -> list of messages (one conversation each)

def compact(messages):
    """Summarize old turns once the history reaches the trigger; otherwise return it unchanged."""
    update = summarizer.before_model({"messages": messages}, runtime=None)
    return messages if update is None else update["messages"][1:]   # [0] is a "remove all" marker

### Chat interface
ipywidgets front end (LangChain provides no notebook UI). Answers are streamed with
`answer_chain.stream()`: `ChatHuggingFace` generates in a background thread through a
`TextIteratorStreamer`, and the reply is re-rendered as Markdown about 10 times per second.

| Step | Model | Display |
|---|---|---|
| Rewrite the follow-up (skipped on the first turn) | 2B | *searched for* line |
| `retrieval_chain`: HyDE → hybrid search → re-rank | 2B, Weaviate, cross-encoder | status line |
| Abstain gate, then `answer_chain.stream(...)` | 9B | streamed answer |
| Sources | n/a | collapsible list |
| Store the turn, summarize older turns if needed | 2B | after the answer is displayed |

The status line reports rewrite, retrieval, time to first token and total time. **New chat** clears the session.

In [ ]:
def bubble(content_html, role):
    """A chat bubble. Semi-transparent colours so it reads well in light and dark themes."""
    style = ("background: rgba(59,130,246,.18); margin-left: auto;" if role == "user"
             else "background: rgba(127,127,127,.12);")
    return (f'<div style="{style} padding: 8px 14px; border-radius: 14px; max-width: 85%; width: fit-content; '
            f'line-height: 1.5; overflow-wrap: anywhere;">{content_html}</div>')

class ChatUI:
    """Notebook chat window over the RAG pipeline: streamed answers + conversation memory."""

    def __init__(self, session_id="ui"):
        self.session_id, self.busy = session_id, False
        self.log = widgets.VBox(layout=widgets.Layout(width="100%"))
        self.status = widgets.HTML()
        self.input = widgets.Text(placeholder="Ask about CV / NLP papers…  (Enter to send)",
                                  continuous_update=False, layout=widgets.Layout(flex="1"))
        self.send_button = widgets.Button(description="Send", button_style="primary")
        new_button = widgets.Button(description="New chat")
        self.input.observe(lambda change: self.ask(change["new"]), names="value")   # fires on Enter
        self.send_button.on_click(lambda _: self.ask(self.input.value))
        new_button.on_click(lambda _: self.reset())
        header = widgets.HBox([widgets.HTML("<b>ArxivSage</b> · chat with 122k CV / NLP arXiv papers"), new_button],
                              layout=widgets.Layout(justify_content="space-between"))
        self.widget = widgets.VBox([header, self.log, self.status, widgets.HBox([self.input, self.send_button])],
                                   layout=widgets.Layout(border="1px solid rgba(127,127,127,.4)", padding="10px"))

    def _add(self, value):
        w = widgets.HTML(value, layout=widgets.Layout(width="100%"))
        self.log.children = (*self.log.children, w)
        return w

    def _status(self, text):
        self.status.value = f'<span style="font-size: 12px; opacity: .7;">{text}</span>'

    def reset(self):
        sessions.pop(self.session_id, None)
        self.log.children = ()
        self._status("new conversation")

    def ask(self, question):
        question = question.strip()
        if not question or self.busy:
            return
        self.busy = self.send_button.disabled = self.input.disabled = True
        self.input.value = ""
        try:
            self._answer(question)
        except Exception as e:
            self._status(f'<span style="color: #e5484d;">error: {html.escape(repr(e))}</span>')
        finally:
            self.busy = self.send_button.disabled = self.input.disabled = False

    def _answer(self, question):
        t0, timings = time.time(), {}
        self._add(bubble(html.escape(question), "user"))

        history = sessions.get(self.session_id, [])
        standalone = question
        if history:
            self._status("rewriting the follow-up…")
            standalone = condense_chain.invoke({"history": get_buffer_string(history), "question": question}).strip()
            timings["rewrite"] = time.time() - t0
            if standalone != question:
                self._add(f'<div style="font-size: 12px; opacity: .7; margin: 0 6px;">searched for: <i>{html.escape(standalone)}</i></div>')

        self._status("searching: HyDE → hybrid search → re-ranking…")
        t = time.time()
        state = retrieval_chain.invoke({"question": standalone})
        timings["retrieval"] = time.time() - t

        reply = self._add(bubble("<i>…</i>", "bot"))
        if is_weak(state):
            answer, sources = ABSTAIN_MESSAGE, []
        else:
            self._status("answering…")
            answer, sources, last_render = "", state["sources"], 0.0
            for token in answer_chain.stream(state):          # LangChain streaming, token by token
                if not answer:
                    timings["first token"] = time.time() - t0
                answer += token
                if time.time() - last_render > 0.1:            # re-render ~10x per second
                    reply.value, last_render = bubble(mistune.html(answer), "bot"), time.time()
        reply.value = bubble(mistune.html(answer), "bot")

        if sources:
            items = "".join(f'<li><a href="{d.metadata["url"]}" target="_blank">{html.escape(d.metadata["title"])}</a>'
                            f' · {d.metadata["published"]} · rerank {d.metadata["rerank_score"]:.2f}</li>' for d in sources)
            self._add(f'<details style="margin: 0 6px 10px;"><summary style="cursor: pointer; font-size: 13px;">'
                      f'Sources ({len(sources)})</summary><ol style="font-size: 13px;">{items}</ol></details>')
        timings["total"] = time.time() - t0
        summary = " · ".join(f"{k} {v:.1f}s" for k, v in timings.items())

        # Memory is updated after the answer is on screen: the summary call never delays a reply.
        self._status(summary + " · updating memory…")
        titles = "; ".join(f"[{i}] {d.metadata['title']}" for i, d in enumerate(sources, 1))
        stored = answer + (f"\n\nSources: {titles}" if titles else "")
        sessions[self.session_id] = compact(history + [HumanMessage(question), AIMessage(stored)])
        self._status(summary)

ui = ChatUI()
display(ui.widget)

Example conversation:
1. *How do recent papers reduce hallucination in vision-language models?*
2. *Which of those methods don't need any extra training?* (follow-up, rewritten)
3. *Only show me ones from 2026.* (rewritten with the topic; "2026" becomes a year filter)

The summarizer triggers after the third turn. Current session memory:

In [ ]:
for msg in sessions.get(ui.session_id, []):
    print(f"--- {msg.type} ---\n{msg.content[:600]}\n")

## 11. Comparison with notebook 03

| | Notebook 03 | Notebook 04 |
|---|---|---|
| Pipeline code | ~150 lines | ~130 lines |
| Provided by the framework | n/a | prompt templating, JSON parsing with fallback, chain composition |
| Custom code in both | n/a | HyDE averaging, filters, scored re-ranking, per-paper de-duplication |
| Implicit behaviour | none | thinking mode, auto-schema, `CrossEncoderReranker` discarding scores |
| Switching to an API LLM | edit `llm_chat()` | replace one constructor (`ChatOpenAI(...)`, etc.) |
| Tracing, streaming, async | not implemented | built in (`.stream()`, `.batch()`, LangSmith) |
| Chat memory | not implemented | ~35 lines, with summarization of older turns |
| Streaming chat UI | not implemented | `answer_chain.stream()` + ~90 lines of ipywidgets |

Both implementations share the same retrieval design. LangChain simplifies wiring, prompt handling and
component swaps; the retrieval-specific logic is custom code in both.

**Shutdown.** `client.close()` closes the Weaviate connection and stops embedded Weaviate if this kernel
started it. It is not a code cell, so that *Run All* does not close the connection under the chat interface.
```python
client.close()
```